# Modul 12 | Python Unit Tests | TDD i dobre praktyki | v2.0 | Kamil Bartocha


## Rozklad jazdy
- 🔹 1. TDD w praktyce - Red-Green-Refactor krok po kroku
- 🔹 2. Organizacja testow i wzorce (Object Mother, Builder)
- 🔹 3. CI/CD, tox i testowanie na wielu wersjach


## 1. 🔹 TDD w praktyce - Red-Green-Refactor krok po kroku

Programowanie sterowane testami (TDD - Test-Driven Development) to
technika, w ktorej testy piszemy **przed** kodem produkcyjnym.
Cykl TDD sklada sie z trzech krokow:

```
RED   → napisz test, ktory nie przechodzi (brak implementacji)
GREEN → napisz **minimalna** implementacje, ktory sprawia, ze test przechodzi
REFACTOR → popraw kod (bez zmiany zachowania), testy musza nadal przechodzic
```

**Zasady TDD:**
- Nie pisz kodu produkcyjnego bez nieprzechodzacego testu (Red).
- Pisz tylko tyle kodu, ile potrzeba do przejscia testu (Green).
- Refaktoryzuj, gdy testy sa zielone (Refactor).

**Zalety TDD:**
- Projekt API powstaje zanim powstanie implementacja.
- 100% pokrycie jest naturalnym efektem procesu.
- Zmiany sa bezpieczne - testy chwytaja regresje.

**Kiedy stosowac TDD:**
- Nowe funkcjonalnosci z jasno okreslonymi wymaganiami.
- Naprawianie bledow: najpierw test reprodukujacy blag, potem poprawka.
- Refaktoryzacja - testy stanowia siatke bezpieczenstwa.


In [ ]:
import subprocess, tempfile, os, textwrap

def _run(code_str, flags="-v"):
    """Zapisuje kod do pliku tymczasowego i uruchamia pytest."""
    with tempfile.NamedTemporaryFile(
        suffix="_test.py", mode="w", delete=False
    ) as f:
        f.write(textwrap.dedent(code_str))
        name = f.name
    try:
        result = subprocess.run(
            ["python3", "-m", "pytest", name, flags, "--tb=short", "-q"],
            capture_output=True, text=True
        )
        output = result.stdout + result.stderr
        print(output[:3000])
    finally:
        os.unlink(name)


# ── Przyklad: cykl Red → Green → Refactor dla Counter ─────────────────────
# KROK 1: RED — test nie przechodzi (brak implementacji)
step1_red = '''
import pytest

class Counter:
    pass  # placeholder - celowo pusta implementacja

def test_counter_initial_value_is_zero():
    c = Counter()
    # brak atrybutu value -> AttributeError (RED)
    assert c.value == 0

def test_counter_increment_increases_value():
    c = Counter()
    # brak metody increment -> AttributeError (RED)
    c.increment()
    assert c.value == 1
'''
print("=== KROK 1: RED ===")
_run(step1_red)

# KROK 2: GREEN — minimalna implementacja
step2_green = '''
class Counter:
    def __init__(self):
        self.value = 0

    def increment(self):
        self.value += 1

    def reset(self):
        self.value = 0

def test_counter_initial_value_is_zero():
    c = Counter()
    assert c.value == 0

def test_counter_increment_increases_value():
    c = Counter()
    c.increment()
    assert c.value == 1

def test_counter_increment_multiple_times():
    c = Counter()
    c.increment()
    c.increment()
    c.increment()
    assert c.value == 3

def test_counter_reset():
    c = Counter()
    c.increment()
    c.reset()
    assert c.value == 0
'''
print("=== KROK 2: GREEN ===")
_run(step2_green)


---

### 🐍 Ćwiczenia — TDD Red-Green-Refactor

1. Zaprojektuj `BankAccount` przez TDD - zacznij od testu, napisz minimum kodu
2. Rozszerz `BankAccount` przez TDD o `transfer(to_account, amount)`
3. *(Trudniejsze)* TDD dla `ShoppingCart` z rabatem i podatkiem VAT


In [ ]:
# Ćwiczenie 1: BankAccount przez TDD
# Postepuj krok po kroku:
#   RED:     napisz test, który nie przechodzi
#   GREEN:   dodaj minimalna implementacje
#   REFACTOR: popraw kod zachowując testy na zielono

bank_tdd = '''
import pytest

# ── TWOJA IMPLEMENTACJA (wypełnij po napisaniu testów) ───────────────────
class BankAccount:
    def __init__(self, owner, balance=0):
        ...

    def deposit(self, amount):
        ...

    def withdraw(self, amount):
        ...

    @property
    def balance(self):
        ...


# ── TESTY (napisz je PRZED implementacją - TDD) ───────────────────────────
def test_bank_account_initial_balance_is_zero():
    account = BankAccount("Alice")
    ...

def test_bank_account_deposit_increases_balance():
    account = BankAccount("Alice")
    account.deposit(100)
    ...

def test_bank_account_withdraw_decreases_balance():
    account = BankAccount("Alice", balance=200)
    account.withdraw(50)
    ...

def test_bank_account_withdraw_raises_when_insufficient_funds():
    account = BankAccount("Alice", balance=100)
    # hint: użyj pytest.raises(ValueError)
    ...

def test_bank_account_owner_stored_correctly():
    account = BankAccount("Bob")
    ...
'''
_run(bank_tdd)


In [ ]:
# Ćwiczenie 2: Rozszerz BankAccount o transfer() przez TDD
# Najpierw napisz testy dla transfer(), potem implementacje.

transfer_tdd = '''
import pytest

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Amount must be positive")
        self._balance += amount

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

    @property
    def balance(self):
        return self._balance

    def transfer(self, target, amount):
        # hint: użyj withdraw i deposit; błędy powinny być propagowane
        ...


# ── TESTY dla transfer() ─────────────────────────────────────────────────
def test_transfer_moves_funds_between_accounts():
    sender = BankAccount("Alice", balance=500)
    receiver = BankAccount("Bob", balance=100)
    sender.transfer(receiver, 200)
    ...

def test_transfer_raises_when_insufficient_funds():
    sender = BankAccount("Alice", balance=50)
    receiver = BankAccount("Bob")
    # hint: transfer 100 powinien rzucić ValueError
    ...

def test_transfer_zero_amount_raises():
    sender = BankAccount("Alice", balance=100)
    receiver = BankAccount("Bob")
    # hint: transfer 0 lub ujemna kwota powinien rzucić ValueError
    ...
'''
_run(transfer_tdd)


In [ ]:
# Ćwiczenie 3 *(Trudniejsze)*: TDD dla ShoppingCart z rabatem i VAT
# Napisz testy PRZED implementacją. Wymagania:
#   - add_item(name, price, qty=1)
#   - total() zwraca sume bez rabatu i VAT
#   - total_with_vat(rate=0.23) dolacza VAT
#   - apply_discount(percent) obniża kwotę o procent
#   - Rabat i VAT nie mogą byc ujemne

shopping_tdd = '''
import pytest

class ShoppingCart:
    def __init__(self):
        ...

    def add_item(self, name, price, qty=1):
        ...

    def total(self):
        ...

    def total_with_vat(self, rate=0.23):
        # hint: total() * (1 + rate)
        ...

    def apply_discount(self, percent):
        # hint: przechowaj rabat, stosuj w total()
        ...


# ── Testy - napisz PRZED implementacją (TDD) ─────────────────────────────
def test_empty_cart_total_is_zero():
    ...

def test_add_single_item():
    ...

def test_add_multiple_items():
    ...

def test_add_item_with_quantity():
    ...

def test_total_with_vat_default_rate():
    # hint: 100 zł + 23% VAT = 123 zł
    ...

def test_total_with_vat_custom_rate():
    ...

def test_apply_discount_reduces_total():
    # hint: 100 zł, 10% rabat = 90 zł
    ...

def test_apply_negative_discount_raises():
    ...

def test_apply_discount_then_vat():
    # hint: 100 zł, 10% rabat → 90 zł, +23% VAT = 110.70 zł
    ...
'''
_run(shopping_tdd)


## 2. 🔹 Organizacja testow i wzorce (Object Mother, Builder)

Wraz z rozrostem bazy testow, przygotowanie danych testowych staje sie
powtarzalne i kruche. Dwa wzorce rozwiazuja ten problem:

**Object Mother** - klasa fabryczna tworzaca gotowe obiekty testowe
z sensownymi wartosciami domyslnymi. Jeden centralny punkt tworzenia.

```python
class UserMother:
    @staticmethod
    def adult() -> User:
        return User(name="Alice", age=30, email="alice@example.com")

    @staticmethod
    def minor() -> User:
        return User(name="Bob", age=15, email="bob@example.com")
```

**Builder** - plynny interfejs (fluent interface) do budowania obiektow
krok po kroku. Nadaje sie, gdy obiekty maja wiele opcjonalnych pol.

```python
user = UserBuilder().with_name("Alice").with_age(30).build()
```

**Najczestsze anty-wzorce w testach:**

| Anty-wzorzec | Problem | Rozwiazanie |
|---|---|---|
| Test wie za duzo | Assert na polach wewnetrznych | Testuj zachowanie, nie stan |
| Jeden gigantyczny test | Trudny debug, wiele przyczyn bledu | Jeden assert na cel |
| Magiczne wartosci | `assert result == 42` bez kontekstu | Nazwane stale |
| Zaleznosc miedzy testami | Kolejnosc ma znaczenie | Izoluj stan w setUp/fixture |
| Mock wszystkiego | Testy nie sprawdzaja integracji | Mock tylko I/O i zewnetrznych |


In [ ]:
# ── Przyklad: Object Mother i Builder dla klasy Order ────────────────────
from dataclasses import dataclass, field
from typing import List


@dataclass
class Address:
    street: str
    city: str
    zip_code: str


@dataclass
class OrderItem:
    name: str
    price: float
    qty: int = 1


@dataclass
class Order:
    customer: str
    items: List[OrderItem]
    address: Address
    discount: float = 0.0

    def total(self) -> float:
        subtotal = sum(i.price * i.qty for i in self.items)
        return subtotal * (1 - self.discount)


# ── Object Mother ─────────────────────────────────────────────────────────
class OrderMother:
    @staticmethod
    def simple() -> Order:
        """Zamowienie z jedną pozycją, bez rabatu."""
        return Order(
            customer="alice",
            items=[OrderItem("book", 50.0)],
            address=Address("Main St 1", "Warsaw", "00-001"),
        )

    @staticmethod
    def with_discount() -> Order:
        """Zamowienie z 10% rabatem."""
        o = OrderMother.simple()
        o.discount = 0.10
        return o


# ── Builder ───────────────────────────────────────────────────────────────
class OrderBuilder:
    def __init__(self):
        self._customer = "test_customer"
        self._items: List[OrderItem] = []
        self._address = Address("Default St", "City", "00-000")
        self._discount = 0.0

    def for_customer(self, name: str) -> "OrderBuilder":
        self._customer = name
        return self

    def with_item(self, name: str, price: float, qty: int = 1) -> "OrderBuilder":
        self._items.append(OrderItem(name, price, qty))
        return self

    def with_discount(self, pct: float) -> "OrderBuilder":
        self._discount = pct
        return self

    def build(self) -> Order:
        return Order(
            customer=self._customer,
            items=self._items,
            address=self._address,
            discount=self._discount,
        )


# ── Uzycie w testach ──────────────────────────────────────────────────────
simple_order = OrderMother.simple()
print(f"Object Mother  → total: {simple_order.total():.2f}")

built_order = (
    OrderBuilder()
    .for_customer("bob")
    .with_item("laptop", 3000.0)
    .with_item("mouse", 100.0, qty=2)
    .with_discount(0.05)
    .build()
)
print(f"Builder        → customer: {built_order.customer}, "
      f"total: {built_order.total():.2f}")


---

### 🐍 Ćwiczenia — Object Mother, Builder, anty-wzorce

4. Refaktoryzuj testy uzywajac wzorca Object Mother dla `User`
5. Napisz `UserBuilder` z wzorcem Builder dla zlozonych danych testowych
6. *(Trudniejsze)* Zidentyfikuj i napraw anty-wzorce w podanym zestawie testow


In [ ]:
# Ćwiczenie 4: Object Mother dla User
# Poniższe testy powtarzają tworzenie obiektów User.
# Stwórz klasę UserMother z metodami: adult(), minor(), admin(), inactive()
# i zrefaktoryzuj testy tak, by uzywały UserMother.

from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int
    email: str
    role: str = "user"
    active: bool = True

    def is_adult(self) -> bool:
        return self.age >= 18

    def can_admin(self) -> bool:
        return self.role == "admin" and self.active


# ── PRZED refaktoryzacją (powtórzenia danych) ─────────────────────────────
def test_adult_user_before():
    user = User(name="Alice", age=30, email="alice@example.com")
    assert user.is_adult()

def test_minor_user_before():
    user = User(name="Bob", age=15, email="bob@example.com")
    assert not user.is_adult()

def test_admin_can_admin_before():
    user = User(name="Carol", age=35, email="carol@example.com",
                role="admin", active=True)
    assert user.can_admin()


# ── TWOJA IMPLEMENTACJA: UserMother ──────────────────────────────────────
class UserMother:
    @staticmethod
    def adult() -> User:
        ...

    @staticmethod
    def minor() -> User:
        ...

    @staticmethod
    def admin() -> User:
        ...

    @staticmethod
    def inactive() -> User:
        ...


# ── PO refaktoryzacji (używa UserMother) ─────────────────────────────────
def test_adult_user_after():
    user = UserMother.adult()
    assert user.is_adult()

def test_minor_user_after():
    user = UserMother.minor()
    assert not user.is_adult()

def test_admin_can_admin_after():
    user = UserMother.admin()
    ...

def test_inactive_admin_cannot_admin():
    user = UserMother.inactive()
    ...

print("UserMother zaimplementowany!")


In [ ]:
# Ćwiczenie 5: UserBuilder - plynny interfejs do budowania User
# Builder pozwala tworzyć obiekty krok po kroku, nadpisujac tylko
# wybrane pola. Przydatny, gdy obiekt ma wiele opcjonalnych atrybutow.

from dataclasses import dataclass

@dataclass
class User:
    name: str
    age: int
    email: str
    role: str = "user"
    active: bool = True
    phone: str = ""
    address: str = ""


class UserBuilder:
    """Buduje obiekt User z wartościami domyślnymi."""

    def __init__(self):
        # hint: zdefiniuj domyślne wartości wszystkich pól
        self._name = "default_user"
        ...

    def with_name(self, name: str) -> "UserBuilder":
        ...

    def with_age(self, age: int) -> "UserBuilder":
        ...

    def with_email(self, email: str) -> "UserBuilder":
        ...

    def as_admin(self) -> "UserBuilder":
        ...

    def inactive(self) -> "UserBuilder":
        ...

    def with_phone(self, phone: str) -> "UserBuilder":
        ...

    def build(self) -> User:
        ...


# ── Uzycie ────────────────────────────────────────────────────────────────
user1 = UserBuilder().with_name("alice").with_age(30).with_email("a@ex.com").build()
user2 = UserBuilder().as_admin().with_age(25).build()
user3 = UserBuilder().inactive().with_phone("+48 123").build()

print(f"user1: {user1}")
print(f"user2: {user2}")
print(f"user3: {user3}")

assert user1.name == "alice"
assert user2.role == "admin"
assert user3.active == False
print("OK")


In [ ]:
# Ćwiczenie 6 *(Trudniejsze)*: Zidentyfikuj i napraw anty-wzorce
#
# Poniższy zestaw testow zawiera kilka klasycznych anty-wzorcow.
# Zadanie:
#   a) Zidentyfikuj każdy anty-wzorzec (wpisz komentarz opisujący problem)
#   b) Napraw testy, zachowując ich intencje

antipatterns = '''
import pytest

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance
        self._transactions = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Must be positive")
        self._balance += amount
        self._transactions.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        self._transactions.append(("withdraw", amount))

    @property
    def balance(self):
        return self._balance


# ANTY-WZORZEC 1: Gigantyczny test - testuje za wiele rzeczy naraz
account = BankAccount("alice")  # ANTY-WZORZEC 2: stan współdzielony
def test_everything():
    account.deposit(100)
    account.deposit(50)
    account.withdraw(30)
    # ANTY-WZORZEC 3: testuje prywatny atrybut _transactions
    assert account._transactions == [("deposit", 100), ("deposit", 50),
                                      ("withdraw", 30)]
    assert account._balance == 120
    assert account.balance == 120
    try:
        account.withdraw(999)
    except ValueError:
        pass  # ANTY-WZORZEC 4: łapanie wyjątku bez assert


# ANTY-WZORZEC 5: magiczne wartości bez kontekstu
def test_balance():
    # pytanie: skąd 120? trzeba czytać test_everything żeby zrozumieć
    assert account.balance == 120
'''

# hint: zidentyfikuj 5 anty-wzorcow i przepisz testy
fixed = '''
import pytest

class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self._balance = balance
        self._transactions = []

    def deposit(self, amount):
        if amount <= 0:
            raise ValueError("Must be positive")
        self._balance += amount
        self._transactions.append(("deposit", amount))

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        self._transactions.append(("withdraw", amount))

    @property
    def balance(self):
        return self._balance


# NAPRAWA: każdy test jest izolowany przez fixture
import pytest

@pytest.fixture
def account():
    return BankAccount("alice")

# NAPRAWA: jeden cel na test, czytelne nazwy
def test_deposit_increases_balance(account):
    INITIAL = 0
    DEPOSIT_AMOUNT = 100
    account.deposit(DEPOSIT_AMOUNT)
    assert account.balance == INITIAL + DEPOSIT_AMOUNT

def test_withdraw_decreases_balance(account):
    account.deposit(200)
    account.withdraw(50)
    assert account.balance == 150

# NAPRAWA: pytest.raises zamiast try/except bez assert
def test_withdraw_raises_when_insufficient(account):
    with pytest.raises(ValueError, match="Insufficient funds"):
        account.withdraw(999)

# NAPRAWA: testujemy zachowanie (balance), nie prywatny stan (_transactions)
def test_multiple_deposits_accumulate(account):
    account.deposit(100)
    account.deposit(50)
    assert account.balance == 150
'''
_run(fixed)


## 3. 🔹 CI/CD, tox i testowanie na wielu wersjach

Automatyczne uruchamianie testow w pipelinie CI/CD (Continuous Integration)
gwarantuje, ze zmiany nie psuja istniejacego kodu.

**Makefile** - prosty interfejs do zadan deweloperskich:
```make
test:
    pytest tests/ -v

coverage:
    pytest --cov=src --cov-report=term-missing
```

**GitHub Actions** - platforma CI/CD wbudowana w GitHub.
Plik `.github/workflows/test.yml` definiuje pipeline:
```yaml
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: '3.11'}
      - run: pip install -r requirements.txt
      - run: pytest
```

**tox** - narzedzie do testowania na wielu wersjach Pythona i srodowiskach.
Plik `tox.ini`:
```ini
[tox]
envlist = py310, py311

[testenv]
deps = pytest
commands = pytest {posargs}
```

**nox** - alternatywa dla tox, konfiguracja w Pythonie (`noxfile.py`):
```python
import nox

@nox.session(python=["3.10", "3.11"])
def tests(session):
    session.install("-r", "requirements.txt")
    session.run("pytest")
```

**Kiedy uzyc tox vs nox:**
- `tox` - prostsze projekty, konfiguracja INI, szeroko stosowany
- `nox` - bardziej elastyczny, konfiguracja Python, latwy do debugowania


In [ ]:
# ── Przyklad: kompletny setup CI dla projektu Python ─────────────────────
import textwrap

makefile = textwrap.dedent("""
    .PHONY: test coverage lint clean

    test:
        pytest tests/ -v

    coverage:
        pytest --cov=src --cov-report=term-missing --cov-fail-under=85

    lint:
        flake8 src/ tests/ --max-line-length=88

    clean:
        find . -type d -name __pycache__ -exec rm -rf {} +
        find . -name '*.pyc' -delete
        rm -rf .coverage htmlcov/ .pytest_cache/
""").strip()

github_actions = textwrap.dedent("""
    name: Tests

    on:
      push:
        branches: [main, master]
      pull_request:
        branches: [main, master]

    jobs:
      test:
        runs-on: ubuntu-latest
        strategy:
          matrix:
            python-version: ['3.10', '3.11', '3.12']

        steps:
          - uses: actions/checkout@v4

          - name: Set up Python ${{ matrix.python-version }}
            uses: actions/setup-python@v5
            with:
              python-version: ${{ matrix.python-version }}

          - name: Install dependencies
            run: |
              python -m pip install --upgrade pip
              pip install -r requirements.txt

          - name: Run tests
            run: pytest --cov=src --cov-report=xml

          - name: Upload coverage
            uses: codecov/codecov-action@v4
""").strip()

tox_ini = textwrap.dedent("""
    [tox]
    envlist = py310, py311, py312, lint
    isolated_build = True

    [testenv]
    deps =
        pytest
        pytest-cov
    commands =
        pytest {posargs:tests/} --cov=src --cov-report=term-missing

    [testenv:lint]
    deps = flake8
    commands = flake8 src/ tests/

    [flake8]
    max-line-length = 88
    extend-ignore = E203, W503
""").strip()

print("=== Makefile ===")
print(makefile)
print("\n=== .github/workflows/test.yml ===")
print(github_actions)
print("\n=== tox.ini ===")
print(tox_ini)


---

### 🐍 Ćwiczenia — CI/CD, Makefile, tox

7. Napisz `Makefile` z targetami `test`, `coverage`, `lint`
8. Napisz `.github/workflows/test.yml` uruchamiajacy pytest na push
9. *(Trudniejsze)* Skonfiguruj `tox.ini` dla Python 3.10 i 3.11 z `pytest` i `flake8`


In [ ]:
# Ćwiczenie 7: Napisz Makefile z targetami test, coverage, lint, clean
# Wymagania:
#   - test:     uruchamia pytest tests/ -v
#   - coverage: pytest z --cov=src, raport terminal, fail-under=80
#   - lint:     flake8 src/ tests/ --max-line-length=88
#   - clean:    usuwa __pycache__, .pyc, .coverage, htmlcov/

makefile_content = """
.PHONY: test coverage lint clean

test:
    ...

coverage:
    ...

lint:
    ...

clean:
    ...
"""

# ── Wypisz gotowy Makefile ────────────────────────────────────────────────
print(makefile_content)


In [ ]:
# Ćwiczenie 8: Napisz .github/workflows/test.yml
# Wymagania:
#   - Wyzwalany na push i pull_request do main
#   - Uruchamiany na ubuntu-latest
#   - Python 3.11
#   - Kroki: checkout, setup-python, pip install, pytest

workflow_content = """
name: Tests

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      ...
"""

# ── Wypisz gotowy workflow ────────────────────────────────────────────────
print(workflow_content)


In [ ]:
# Ćwiczenie 9 *(Trudniejsze)*: Skonfiguruj tox.ini
# Wymagania:
#   - envlist: py310, py311, lint
#   - [testenv]: instaluje pytest, pytest-cov; uruchamia pytest z --cov
#   - [testenv:lint]: instaluje flake8; sprawdza src/ i tests/
#   - [flake8]: max-line-length=88, extend-ignore=E203
#   - Opcjonalnie: dodaj [testenv:mypy] z mypy src/

tox_content = """
[tox]
envlist = py310, py311, lint
isolated_build = True

[testenv]
deps =
    ...
commands =
    ...

[testenv:lint]
deps = ...
commands = ...

[flake8]
...
"""

# hint: dodaj opcjonalnie sekcję [testenv:mypy]
# hint: użyj {posargs:tests/} żeby umożliwić pytest tests/specific_test.py

print(tox_content)

# ── Noxfile.py jako alternatywa dla tox ──────────────────────────────────
noxfile_content = """
# noxfile.py - alternatywa dla tox, konfiguracja w Pythonie
import nox

@nox.session(python=["3.10", "3.11"])
def tests(session):
    session.install("-r", "requirements.txt")
    session.run("pytest", "tests/", "--cov=src")

@nox.session
def lint(session):
    session.install("flake8")
    session.run("flake8", "src/", "tests/")
"""
print(noxfile_content)
